# Gemini on REAL sky images

The counterpart of `gemini_withapi_test.ipynb`, run against real published
figures instead of synthetic ones. Gemini only, because it was the most
accurate of the three on the synthetic benchmark.

Questions and ground truth come from `VQA_realskyimages/`, built by
`create_vqa_from_datas_realsky.ipynb`. Three things differ from the synthetic
run, and all three matter:

**Five questions, not 21-25.** For a figure we did not draw there is no ground
truth for colormap, plot style, colour statistics or the underlying
distribution. What remains is geometry and position: `field width`,
`field height`, `pixel scale`, `center right ascension`, `center declination`.
The first three are worded **identically** to the synthetic benchmark, so those
scores are directly comparable; the two positional ones are new.

**The images sent are stripped copies.** Every AstroExplorer JPEG carries the
AVM astrometry in an XMP packet -- the answer key, inside the file. This
notebook reads `real_data/images_nometa/`, which holds re-encoded,
metadata-free versions. Pointing `imgs_dir` at the original `images/` would
quietly invalidate the whole run.

**All questions sit at figure level.** A real published figure is one image,
so there is no per-panel nesting and no `Plot-level questions` block. The QA
assembly below walks whatever levels are present rather than assuming
`Level 1` exists -- with the default question set there is no Level 1 at all.


In [1]:
# ============================ RUN CONFIG ============================
# Same model and settings as the synthetic low-tier gemini run, so the two are
# comparable.  See gemini_withapi_test.ipynb for why these specific values.
out_base = '~/astro_sky_image_vqa/LMM_outputs_realskyimages/'

dir_api = out_base + 'gemini/'
model_name = 'gemini-3.5-flash-lite'
thinking_level = 'minimal'     # matches the low tier used on the synthetic set

key_file = '/Users/jnaiman/.gemini/key.txt'

# ---- where the real-sky VQA lives ----
jsons_dir = '~/astro_sky_image_vqa/VQA_realskyimages/qa_jsons/'

# METADATA-STRIPPED images.  Do NOT point this at real_data/images/ -- those
# carry the AVM astrometry (the answers) in an XMP packet, and a model that
# reads file metadata would score perfectly without looking at the picture.
imgs_dir = '/Users/jnaiman/Dropbox/WASP2026/real_data/images_nometa/'

img_format = 'jpeg'
tmp_dir = '~/Downloads/tmp/'

# Skip figures whose pickle already exists, so an interrupted run resumes
# instead of re-paying for finished work.
restart = False

# None = every figure in jsons_dir (156).  Set an int to cap for a quick test.
iMax = None

# Ask for reasoning alongside the answer, identical to the synthetic gemini run
# so the responses have the same shape and the same parser works on both.
reasoning_text = 'In addition to providing your answer, please provide your reasoning.  Include this as a separate JSON snippet formatted as {"explanation":""} and fill in your reasoning as a string.  Your answer should only include the answer JSON and the explanation JSON snippets, no other text.'


In [2]:
import base64
from PIL import Image
import numpy as np
import json
import re
import pickle
import os
from glob import glob
# import google.generativeai as genai
from google import genai
from google.genai import types
# typed errors, so a 429 can be told apart from a 400 (see send_to_gemini)
from google.genai.errors import ClientError, ServerError

# debug
from importlib import reload
from copy import deepcopy


from sys import path
path.append('../')
import utils.llm_utils
reload(utils.llm_utils)
from utils.llm_utils import parse_qa, load_image, get_img_json_pair, parse_for_errors

import time
from utils.plot_qa_utils import get_nplots

# --- expand ~ ONCE, and assign back ----------------------------------------
# These must be rebound, not expanded locally: everything downstream (glob,
# open, os.path.join) uses the variables directly, and glob('~/...') silently
# matches nothing rather than erroring.
out_base  = os.path.expanduser(out_base)
dir_api   = os.path.expanduser(dir_api)
key_file  = os.path.expanduser(key_file)
jsons_dir = os.path.expanduser(jsons_dir)
imgs_dir  = os.path.expanduser(imgs_dir)
tmp_dir   = os.path.expanduser(tmp_dir)

# --- make the directories we WRITE to -------------------------------------
# dir_api holds one <id>_qa.pickle per figure; tmp_dir holds the resized images
# load_image() falls back to when a request is too large.  Neither is created
# by anything upstream, so a fresh out_base fails on the first save.
for _d in (dir_api, tmp_dir):
    if not os.path.exists(_d):
        os.makedirs(_d, exist_ok=True)   # makedirs, not mkdir: the path is nested
        print('made:', _d)

# --- check the directories we READ from ------------------------------------
for _name, _d in (('jsons_dir', jsons_dir), ('imgs_dir', imgs_dir)):
    if not os.path.isdir(_d):
        print('[WARN] %s does not exist: %s' % (_name, _d))
if not os.path.exists(key_file):
    print('[WARN] key_file does not exist:', key_file)


In [3]:
import os, json, pickle
from glob import glob

out_base  = os.path.expanduser(out_base)
dir_api   = os.path.expanduser(dir_api)
key_file  = os.path.expanduser(key_file)
jsons_dir = os.path.expanduser(jsons_dir)
imgs_dir  = os.path.expanduser(imgs_dir)
tmp_dir   = os.path.expanduser(tmp_dir)

for _d in (dir_api, tmp_dir):
    os.makedirs(_d, exist_ok=True)

for _n, _d in (('jsons_dir', jsons_dir), ('imgs_dir', imgs_dir)):
    if not os.path.isdir(_d):
        print('[WARN] %s does not exist: %s' % (_n, _d))
if not os.path.exists(key_file):
    print('[WARN] key_file does not exist:', key_file)

# --- guard: refuse to run against images that still carry the answer key ----
# Cheap, and it catches the one configuration mistake that would silently
# invalidate every number this notebook produces.
def _metadata_bytes(path):
    raw = open(path, 'rb').read()
    return raw.count(b'avm:') + raw.count(b'<x:xmpmeta') + raw.count(b'Spatial.')

_probe = sorted(glob(os.path.join(imgs_dir, '*.' + img_format)))
if not _probe:
    raise SystemExit('no %s images in %s -- run create_vqa_from_datas_realsky.ipynb '
                     'or VQA_realskyimages/fetch_images.py first' % (img_format, imgs_dir))
_dirty = [p for p in _probe[:25] if _metadata_bytes(p)]
if _dirty:
    raise SystemExit('THESE IMAGES STILL CARRY AVM METADATA -- refusing to run:\n  '
                     + '\n  '.join(os.path.basename(p) for p in _dirty[:5]))
print('images: %d, metadata-free (checked first %d)' % (len(_probe), min(25, len(_probe))))
print('output: %s' % dir_api)

images: 156, metadata-free (checked first 25)
output: /Users/jnaiman/astro_sky_image_vqa/LMM_outputs_realskyimages/gemini/


In [4]:
# setup
with open(key_file,'r') as f:
    api_key = f.read()

# Configure the API key
client = genai.Client(api_key=api_key.strip())

# NOTE: previously a GenerateContentConfig was built here with ONLY
# thinking_config and passed into send_to_gemini().  Because send_to_gemini only
# fills in system_instruction when config is None, passing this meant the
# persona/system prompt was silently never sent to Gemini -- while ChatGPT and
# Claude both received it.  Leave this as None so the per-question config built
# inside send_to_gemini (which does set system_instruction) is used.
config = None

# thinking level is applied per-call inside send_to_gemini


In [5]:
#model_gemini = genai.GenerativeModel(model)

In [6]:
# ================= WHAT TO RUN =================
# Every figure in the real-sky set.  No category sampling: there are only 156
# and they are not categorised -- the 7 with untrustworthy astrometry were
# already dropped when the VQA was built.
jsons_to_parse = sorted(glob(os.path.join(jsons_dir, '*_qa.json')))
print('qa jsons found:', len(jsons_to_parse))

def _img_for(json_path):
    vid = os.path.basename(json_path).removesuffix('_qa.json')
    return os.path.join(imgs_dir, vid + '.' + img_format)

missing = [j for j in jsons_to_parse if not os.path.exists(_img_for(j))]
if missing:
    print('[WARN] %d qa jsons have no stripped image and will be skipped' % len(missing))
    for j in missing[:5]:
        print('   ', os.path.basename(j))
jsons_to_parse = [j for j in jsons_to_parse if os.path.exists(_img_for(j))]

already = [j for j in jsons_to_parse
           if os.path.exists(dir_api + os.path.basename(j).removesuffix('.json') + '.pickle')]
print('to ask        :', len(jsons_to_parse))
print('already done  :', len(already), '(skipped, restart=False)' if not restart else '(will be REDONE)')
print('this run will send:', len(jsons_to_parse) - (0 if restart else len(already)), 'figures')

# how many questions that is
_nq = 0
for j in jsons_to_parse:
    rec = json.loads(json.load(open(j)))
    _nq += sum(len(k) for lv in rec['VQA'].values() for k in lv.values())
print('questions in the full set:', _nq)

qa jsons found: 156
to ask        : 156
already done  : 0 (skipped, restart=False)
this run will send: 156 figures
questions in the full set: 780


In [7]:
def send_to_gemini(question_list, image_path, client,model_name='gemini-3.5-flash-lite',
                    test_run = True, 
                    verbose=True, verbose_tokens = False,
                    system_prompt = None, img_format='jpg',         
                   large_image= False, config=None, reasoning=None,
                   thinking_level=None,
                   max_retries=10, sleep_time=1):
    """
    Sends an image + question to Gemini.  Does something different for large file, but might be bad idea.

    system_prompt : set to None to use default from question list
    """
    if system_prompt is None:
        system_prompt = question_list['persona']
    if img_format.lower() == 'jpg':
        img_format = 'jpeg'

    err = False
    #model_gemini = genai.GenerativeModel(model)
    #if verbose: print('Model loaded:', model)
    prompt_save = ''; prompt = ''; response = ''
    if not test_run:
        question = question_list['context'] + " " + question_list['question'] + " " + question_list['format']
        if reasoning is not None:
            question += " " + reasoning
        # lowercase the first word, just in case
        question = question.lstrip() # no whitespace
        question = question[0].lower() + question[1:]

        if verbose: print('   on question:',question)
        # Prepare the API request
        prompt = f"I am going to show you an image. Now, {question}"
        prompt_save = f"I am going to show you an image. Now, {question}"
        # --- retry with jittered exponential backoff -----------------------
        # This used to be a bare try/except: any failure, a 429 included, just
        # printed an error and set err, losing that question for this figure.
        # Now rate limits and 5xx are retried the way the claude runner does it.
        # 400s are NOT retried -- waiting does not fix a malformed request.
        attempt = 0
        success = False
        while not success and attempt < max_retries:
            try:
            #if True:

                # # Wait for processing (for video files, you might need to wait longer)
                # if large_image:
                #     # Upload the file to Gemini
                #     #uploaded_file = genai.upload_file(path=image_path)
                #     uploaded_file = client.files.upload(file=image_path)
                #     import time
                #     while uploaded_file.state.name == "PROCESSING":
                #         time.sleep(1)
                #         uploaded_file = genai.get_file(uploaded_file.name)
                # else:
                #     uploaded_file = Image.open(image_path)

                with open(image_path, 'rb') as f:
                    image_bytes = f.read()
                image_part = types.Part.from_bytes(data=image_bytes, mime_type='image/'+img_format)

            
                # Build the config here so system_instruction is ALWAYS applied.
                # (Previously a caller-supplied config skipped this branch entirely
                # and the persona never reached the model.)
                if config is None:
                    cfg_kwargs = {'system_instruction': system_prompt}
                    if thinking_level is not None:
                        # Gemini 3.x: thinking_level, NOT the legacy thinking_budget
                        cfg_kwargs['thinking_config'] = types.ThinkingConfig(
                            thinking_level=thinking_level)
                    config_use = types.GenerateContentConfig(**cfg_kwargs)
                else:
                    # caller supplied one -- make sure it still carries the persona
                    config_use = config
                    if getattr(config_use, 'system_instruction', None) is None:
                        config_use.system_instruction = system_prompt
                # Generate content with the uploaded file
                #response = model_gemini.generate_content([prompt, uploaded_file])
                response = client.models.generate_content(
                    model=model_name,
                    contents=[prompt, image_part],
                    config=config_use
                )
            
                if large_image:
                    # Clean up - delete the uploaded file
                    #genai.delete_file(uploaded_file.name)
                    client.files.delete(name=uploaded_file.name)
            
                #return response.text
        
                success = True

            except ServerError as e:
                # 5xx -- transient on Google's side
                if attempt < max_retries - 1:
                    wait_time = sleep_time*(2 ** attempt) + np.random.uniform(0, 1)
                    print(f"      server error {getattr(e, 'code', '?')}. Waiting {wait_time:.2f}s before retry {attempt + 1}")
                    time.sleep(wait_time)
                    attempt += 1
                else:
                    print('[ERROR]:', str(e))
                    err = True
                    break

            except ClientError as e:
                # 429 is the rate limit; every other 4xx is the caller's fault
                if getattr(e, 'code', None) == 429 and attempt < max_retries - 1:
                    wait_time = sleep_time*(2 ** attempt) + np.random.uniform(0, 1)
                    print(f"      rate limited (429). Waiting {wait_time:.2f}s before retry {attempt + 1}")
                    time.sleep(wait_time)
                    attempt += 1
                else:
                    print('[ERROR]:', str(e))
                    err = True
                    break

            except Exception as e:
                print('[ERROR]:', str(e))
                err = True
                break

            #return f"Error: {str(e)}"
        
    # if not test_run and not err:
    #     for part in response.candidates[0].content.parts:
    #         if not part.thought:  # skip thinking tokens
    #             text = part.text
    if not test_run and not err:
        if response is None:
            print('[ERROR]: response is None')
            err = True
            text = ''
        else:
            text = ''
            for part in response.candidates[0].content.parts:
                if not part.thought:
                    text = part.text
                    break  # take first non-thought part
            if not text:
                print('[WARNING]: no non-thought parts found in response')
        if verbose:
            #print('     response:', str(response.candidates[0].content.parts[0].text).replace('\n',''))
            print('     response:', text)




        question_list['Response (raw)'] = deepcopy(response)
        # Get the response from the API
        #answer = deepcopy(str(response.candidates[0].content.parts[0].text))
        answer = text
        question_list['raw answer'] = answer
        # also calculate usage
        usage = {
            'input_tokens': response.usage_metadata.prompt_token_count,
            'output_tokens': response.usage_metadata.candidates_token_count,
            'total_tokens': response.usage_metadata.total_token_count,
            'cached_content_tokens': getattr(response.usage_metadata, 'cached_content_token_count', 0)
            }

        question_list['usage'] = usage
        if verbose and verbose_tokens:
            print(f"      - Input tokens: {usage.input_tokens}")
            print(f"      - Output tokens: {usage.output_tokens}")
            print(f"      - Total tokens: {usage.input_tokens + usage.output_tokens}")
        # format answer
        if '```json"' in answer:
            answer_format = answer.split('```json"')[-1].split('\n')[0].replace('\n', '')
        elif '```json\n' in answer:
            answer_format = answer.split('```json\n')[-1].split('\n')[0].replace('\n', '')
        elif '```json' in answer:
            answer_format = answer.split('```json')[-1].split('\n')[0].replace('\n', '')
        else:
            answer_format = answer # just give it a shot!
        #answer.replace("```json\n",'').replace("\n```",'')
        try:
            question_list['Response'] = json.loads(answer_format)
        except:
            question_list['Response'] = answer_format
            question_list['Error'] = 'JSON formatting'
        question_list['Response String'] = answer_format
    elif err:
        question_list['raw answer'] = 'ERROR'
    else:
        question_list['Response'] = 'TEST RUN'
        question_list['Response String'] = 'TEST RUN'
        question_list['Response (raw)'] = 'TEST RUN'
    

    return question_list, prompt_save, system_prompt


In [8]:
# ================= RUN =================
reload(utils.llm_utils)
from utils.llm_utils import parse_for_errors

verbose  = False
test_run = False        # True to assemble prompts without calling the API

system_prompt = """You are a helpful assistant that responds only in valid JSON format. Do not include any explanations, reasoning, or text outside of the JSON response."""
if reasoning_text is not None:
    system_prompt = """You are a helpful assistant that responds only in valid JSON format. Do not include any text outside of the requested JSON responses."""


def build_qa(base_json, reasoning_text):
    """
    Flatten the real-sky VQA block into the list of question dicts the sender
    expects.

    Walks whatever levels and kinds are present instead of reaching for
    'Level 1' -> 'Figure-level questions' directly: the real-sky set has no
    Plot-level block at all, and with the default question list it has no
    Level 1 either, so the synthetic notebook's unconditional access would
    raise KeyError on the first figure.
    """
    qa = []
    for level in sorted(base_json['VQA']):
        for kind, questions in base_json['VQA'][level].items():
            for name, v in questions.items():
                qa.append({'Q': v['Q'], 'A': v['A'], 'Level': level, 'type': kind,
                           'Response': "", 'persona': v['persona'],
                           'context': v['context'], 'question': v['question'],
                           'format': v['format'], 'reasoning': reasoning_text,
                           'question name': name})
    return qa


n_total = len(jsons_to_parse) if iMax is None else min(iMax, len(jsons_to_parse))
for ijson, json_path in enumerate(jsons_to_parse):
    if ijson >= n_total:
        continue

    vqa_id   = os.path.basename(json_path).removesuffix('_qa.json')
    out_path = dir_api + os.path.basename(json_path).removesuffix('.json') + '.pickle'
    if os.path.exists(out_path) and not restart:
        continue

    img_path = _img_for(json_path)
    print('on %d of %d: %s' % (ijson + 1, n_total, vqa_id))

    # The real-sky jsons are double-encoded exactly like VQA_full, so this is
    # the same two-step decode get_img_json_pair() does.  Read directly rather
    # than through that helper: it also base64-encodes the image, which
    # send_to_gemini does not use (it uploads the file by path).
    with open(json_path) as f:
        base_json = json.loads(json.load(f))

    qa = build_qa(base_json, reasoning_text)

    for question_list in qa:
        response, prompt, system_prompt_out = send_to_gemini(
            question_list, img_path, client,
            test_run=test_run, verbose=verbose, img_format=img_format,
            system_prompt=system_prompt, config=config, reasoning=reasoning_text,
            model_name=model_name, thinking_level=thinking_level)
        question_list['prompt'] = prompt
        question_list['system prompt'] = system_prompt_out

    qa = parse_for_errors(qa, llm='Gemini')

    if not test_run:
        with open(out_path, 'wb') as ff:
            pickle.dump([qa, model_name], ff)
        print('   saved:', os.path.basename(out_path))
    else:
        print('   would store at:', out_path)

print("!!!!!!!! DONE !!!!!!!!!!!")

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


on 1 of 156: rsky_000001


Q: I am going to show you an image. Now, what is the angular height of the sky region shown in this figure, measured along the declination axis? Give the value in arcminutes. Please format the output as a json as {"field height":""} for this figure, where the value should be a float, expressed in arcminutes. In addition to providing your answer, please provide your reasoning.  Include this as a separate JSON snippet formatted as {"explanation":""} and fill in your reasoning as a string.  Your answer should only include the answer JSON and the explanation JSON snippets, no other text.
Gemini A: {'explanation': "The vertical axis represents the declination (delta) in 1950 coordinates, showing tick marks at 16', 20', 24', and 28' (implying a declination range from 06 degrees 16' to 06 degrees 28'). The top of the region shown aligns with 16' and the bottom aligns with 28', meaning the angular height along the declination axis is from 16' to 28', spanning a total of 12 arcminutes. However,

Q: I am going to show you an image. Now, what is the angular width of the sky region shown in this figure, measured along the right ascension axis, as a true angle on the sky (i.e. including the cos(declination) factor)? Give the value in arcminutes. Please format the output as a json as {"field width":""} for this figure, where the value should be a float, expressed in arcminutes. In addition to providing your answer, please provide your reasoning.  Include this as a separate JSON snippet formatted as {"explanation":""} and fill in your reasoning as a string.  Your answer should only include the answer JSON and the explanation JSON snippets, no other text.
Gemini A: {'field width': '2.5', 'explanation': 'Looking at the axes, the declination span on the left/right goes from approximately -32 degrees 54 arcminutes to about -32 degrees 51 arcminutes, showing a height of roughly 3 arcminutes. The ticks along the bottom axis for right ascension typically span a similar angular scale on the

Q: I am going to show you an image. Now, what is the right ascension of the centre of this sky image, in the J2000 (FK5) frame? Note that the axis labels, if present, may be given in a different equinox such as B1950; give your answer in J2000. Give the value in decimal degrees. Please format the output as a json as {"center right ascension":""} for this figure, where the value should be a float, expressed in decimal degrees J2000. In addition to providing your answer, please provide your reasoning.  Include this as a separate JSON snippet formatted as {"explanation":""} and fill in your reasoning as a string.  Your answer should only include the answer JSON and the explanation JSON snippets, no other text.
Gemini A: {'center right ascension': '29.4121', 'explanation': 'The image shows Right Ascension labeled in B1950 at the bottom, spanning from 1h 55m 50s down to 1h 55m 30s. The left edge corresponds to roughly 1h 55m 52.5s and the right edge to roughly 1h 55m 27.5s, placing the cent

...

Q: I am going to show you an image. Now, what is the angular width of the sky region shown in this figure, measured along the right ascension axis, as a true angle on the sky (i.e. including the cos(declination) factor)? Give the value in arcminutes. Please format the output as a json as {"field width":""} for this figure, where the value should be a float, expressed in arcminutes. In addition to providing your answer, please provide your reasoning.  Include this as a separate JSON snippet formatted as {"explanation":""} and fill in your reasoning as a string.  Your answer should only include the answer JSON and the explanation JSON snippets, no other text.
Gemini A: {'field width': '10.0', 'explanation': "The right ascension axis ranges from 17h 42m 45s on the left to 17h 42m 15s on the right. This corresponds to a time difference of 30 seconds of time (ΔRA = 30s). To convert this to a true angular width on the sky, we use the relation: Width = ΔRA * 15 * cos(Dec), where Dec ≈ -29 deg

Q: I am going to show you an image. Now, what is the angular width of the sky region shown in this figure, measured along the right ascension axis, as a true angle on the sky (i.e. including the cos(declination) factor)? Give the value in arcminutes. Please format the output as a json as {"field width":""} for this figure, where the value should be a float, expressed in arcminutes. In addition to providing your answer, please provide your reasoning.  Include this as a separate JSON snippet formatted as {"explanation":""} and fill in your reasoning as a string.  Your answer should only include the answer JSON and the explanation JSON snippets, no other text.
Gemini A: {'explanation': "The right ascension axis ranges from 03h 38m 20s on the left to 03h 38m 05s on the right, which is a total span of 15 seconds of time. At a declination of approximately -24 degrees 43 minutes, 1 second of right ascension corresponds to 15 * cos(-24.72 degrees) arcseconds. Therefore, 15 seconds of RA equals

Q: I am going to show you an image. Now, what is the angular width of the sky region shown in this figure, measured along the right ascension axis, as a true angle on the sky (i.e. including the cos(declination) factor)? Give the value in arcminutes. Please format the output as a json as {"field width":""} for this figure, where the value should be a float, expressed in arcminutes. In addition to providing your answer, please provide your reasoning.  Include this as a separate JSON snippet formatted as {"explanation":""} and fill in your reasoning as a string.  Your answer should only include the answer JSON and the explanation JSON snippets, no other text.
Gemini A: {'explanation': "Looking at the right ascension axis at the bottom of the map, the labels go from 05h 33m 08s on the left to 05h 33m 02s on the right. This represents a total difference in right ascension of 6 seconds of time (from 08s to 02s). To convert seconds of time to arcseconds on the sky, we use the relation that 1

Q: I am going to show you an image. Now, what is the angular height of the sky region shown in this figure, measured along the declination axis? Give the value in arcminutes. Please format the output as a json as {"field height":""} for this figure, where the value should be a float, expressed in arcminutes. In addition to providing your answer, please provide your reasoning.  Include this as a separate JSON snippet formatted as {"explanation":""} and fill in your reasoning as a string.  Your answer should only include the answer JSON and the explanation JSON snippets, no other text.
Gemini A: {'explanation': "The image shows a declination axis ranging from 59°46' at the bottom to 60°04' (or more precisely, with the tick marks every 4 arcminutes: 60°02', 59°58', 59°54', 59°50', 59°46'). The top border aligns roughly with 60°04' (extending just past 60°02') and the bottom border is at 59°46'. The difference between the top and bottom declination limits is approximately 60°04' - 59°46' =

Q: I am going to show you an image. Now, what is the angular height of the sky region shown in this figure, measured along the declination axis? Give the value in arcminutes. Please format the output as a json as {"field height":""} for this figure, where the value should be a float, expressed in arcminutes. In addition to providing your answer, please provide your reasoning.  Include this as a separate JSON snippet formatted as {"explanation":""} and fill in your reasoning as a string.  Your answer should only include the answer JSON and the explanation JSON snippets, no other text.
Gemini A: {'explanation': "Looking at the vertical axis (Declination, J2000) of the map, the tick marks range from 29 degrees 54 minutes 15 seconds at the top to 29 degrees 53 minutes 15 seconds near the bottom (or specifically spanning from Dec = 29d 54m 15s down to 29d 53m 15s, which is 1 minute of arc, but checking the exact labels: top is 29 54 15, next is 00 (meaning 29 54 00), 53 45, 53 30, 53 15. Wa

## Look at the results

In [9]:
import glob as _glob
import pandas as pd
from utils.parse_lmm_output_utils import answer_for, expected_key_from_format

pickles = sorted(_glob.glob(dir_api + '*.pickle'))
print('pickles:', len(pickles))

rows = []
for p in pickles:
    qa, mid = pickle.load(open(p, 'rb'))
    vid = os.path.basename(p).removesuffix('_qa.pickle')
    for e in qa:
        name = e.get('question name') or list(e['A'])[0]
        truth = list(e['A'].values())[0] if isinstance(e['A'], dict) else e['A']
        got = answer_for(e.get('raw answer') or e.get('Response'),
                         expected_key_from_format(e.get('format')))
        rows.append({'vqa_id': vid, 'question': name, 'truth': truth, 'answer': got,
                     'parsed': got is not None})
df = pd.DataFrame(rows)
if len(df):
    print('\nanswers: %d   unparsable: %d (%.2f%%)'
          % (len(df), (~df['parsed']).sum(), 100 * (~df['parsed']).mean()))
    display(df.groupby('question')['parsed'].agg(['count', 'sum']).rename(
        columns={'sum': 'parsed'}))
df.head(10)

pickles: 156

answers: 780   unparsable: 1 (0.13%)


,count,parsed
question,,
center declination,156,156
center right ascension,156,156
field height,156,156
field width,156,155
pixel scale,156,156


,vqa_id,question,truth,answer,parsed
0,rsky_000001,field width,18.892400,40.97,True
1,rsky_000001,field height,21.529000,16.0,True
2,rsky_000001,pixel scale,1.138100,1.7,True
3,rsky_000001,center right ascension,91.942757,91.890,True
4,rsky_000001,center declination,-6.384538,-6.381,True
5,rsky_000002,field width,8.512700,2.5,True
6,rsky_000002,field height,6.082900,3.3,True
7,rsky_000002,pixel scale,0.492500,0.5,True
8,rsky_000002,center right ascension,9.029981,203.875,True
9,rsky_000002,center declination,-32.599800,-32.8906,True


### Accuracy on the three questions shared with the synthetic set

`field width`, `field height` and `pixel scale` are worded identically here and
in `VQA_full`, so these numbers sit beside the synthetic ones directly. All
three span orders of magnitude (pixel scale runs 0.012 to 45 arcsec/px), so
they are scored as a log-ratio rather than a percentage -- the same choice
`mean_med_calcs_figure.ipynb` makes, and for the same reason.


In [10]:
import numpy as np

SHARED = ['field width', 'field height', 'pixel scale']
sub = df[df['question'].isin(SHARED)].copy() if len(df) else df

def _num(v):
    try:
        return float(str(v).strip().replace(',', ''))
    except Exception:
        return np.nan

if len(sub):
    sub['lmm'] = sub['answer'].map(_num)
    sub['gt'] = sub['truth'].map(_num)
    ok = np.isfinite(sub['lmm']) & np.isfinite(sub['gt']) & (sub['lmm'] > 0) & (sub['gt'] > 0)
    sub['logratio'] = np.where(ok, np.log10(sub['lmm'].where(ok) / sub['gt'].where(ok)), np.nan)

    out = []
    for q, g in sub.groupby('question'):
        lr = g['logratio'].dropna()
        out.append({'question': q, 'answers': len(g), 'scored': len(lr),
                    'median dex': lr.median() if len(lr) else np.nan,
                    'within 2x %': 100 * (lr.abs() < np.log10(2)).mean() if len(lr) else np.nan,
                    'within 10x %': 100 * (lr.abs() < 1).mean() if len(lr) else np.nan,
                    'over %': 100 * (lr > 0).mean() if len(lr) else np.nan})
    display(pd.DataFrame(out).set_index('question').round(2))
else:
    print('no results yet -- run the cell above first')

,answers,scored,median dex,within 2x %,within 10x %,over %
question,,,,,,
field height,156,156,-0.23,67.95,90.38,17.31
field width,156,155,-0.26,36.13,94.84,22.58
pixel scale,156,156,0.37,36.54,88.46,86.54


### Position: how close did it get, on the sky

The positional questions have no synthetic counterpart, and a degree error in
RA means something different at the pole than at the equator. So they are
scored as a true angular separation between the answered position and the real
one, rather than per-coordinate.

Worth reading alongside the figure's own field size: landing 2 degrees away is
a miss on an arcminute field and a near-hit on a 39-degree one.


In [11]:
from astropy.coordinates import SkyCoord
import astropy.units as u

if len(df):
    piv = df[df['question'].isin(['center right ascension', 'center declination'])] \
            .pivot_table(index='vqa_id', columns='question',
                         values=['truth', 'answer'], aggfunc='first')
    recs = []
    for vid, r in piv.iterrows():
        try:
            ra_t, dec_t = float(r[('truth', 'center right ascension')]), float(r[('truth', 'center declination')])
            ra_a, dec_a = _num(r[('answer', 'center right ascension')]), _num(r[('answer', 'center declination')])
            if not (np.isfinite(ra_a) and np.isfinite(dec_a)):
                recs.append({'vqa_id': vid, 'sep_deg': np.nan}); continue
            sep = SkyCoord(ra_t * u.deg, dec_t * u.deg).separation(
                  SkyCoord((ra_a % 360) * u.deg, np.clip(dec_a, -90, 90) * u.deg)).deg
            recs.append({'vqa_id': vid, 'sep_deg': sep})
        except Exception:
            recs.append({'vqa_id': vid, 'sep_deg': np.nan})
    pos = pd.DataFrame(recs)
    good = pos['sep_deg'].dropna()
    print('figures with a usable position answer: %d of %d' % (len(good), len(pos)))
    if len(good):
        print('  median separation : %.2f deg' % good.median())
        print('  within 1 deg      : %d (%.0f%%)' % ((good < 1).sum(), 100 * (good < 1).mean()))
        print('  within 10 deg     : %d (%.0f%%)' % ((good < 10).sum(), 100 * (good < 10).mean()))
        print('  beyond 90 deg     : %d (%.0f%%)' % ((good > 90).sum(), 100 * (good > 90).mean()))
    display(pos.sort_values('sep_deg').head(10))

figures with a usable position answer: 154 of 156
  median separation : 0.45 deg
  within 1 deg      : 108 (70%)
  within 10 deg     : 136 (88%)
  beyond 90 deg     : 3 (2%)


,vqa_id,sep_deg
152,rsky_000153,0.004368
117,rsky_000118,0.005616
110,rsky_000111,0.006680
155,rsky_000156,0.007245
12,rsky_000013,0.007842
141,rsky_000142,0.009276
78,rsky_000079,0.014943
120,rsky_000121,0.016214
35,rsky_000036,0.016979
77,rsky_000078,0.017861
